# LangChain — Aula 01 · acompanhamento

**Origem:** `Aula01/video01|02|03/main.py` · **Kernel:** o `.venv` **deste módulo** (`LangChain_para_aplicacoes_LLM/.venv`), não o da disciplina de NLP.

## O contrato

- **Eu escrevi:** o encanamento — células prontas na ordem, o modelo falso que substitui a API, e a migração do código de agente para a versão atual do LangChain.
- **Você escreve:** o porquê, nos blocos **Sua vez**.

## Tudo aqui roda sem nenhuma chave de API

Esse foi o requisito. A solução tem três camadas, e vale saber em qual você está a cada momento, porque a confiança no resultado muda:

| Camada | O que é real | O que é falso |
|---|---|---|
| **1. Sem modelo nenhum** | tudo — template, ferramentas, schemas | nada |
| **2. Chain com modelo falso** | o encanamento do LCEL inteiro | o texto que o modelo "responde" |
| **3. Agente com modelo falso** | o grafo, o roteamento e a **execução da ferramenta** | só a decisão de *qual* ferramenta chamar |

Na camada 3 a ferramenta roda de verdade: os números que você vai ver são conta do Python, não invenção. O que está roteirizado é a escolha do modelo.

**Vantagem didática inesperada:** sem API o resultado é determinístico. Você roda dez vezes e vê o mesmo loop, o que é melhor para entender o mecanismo do que uma resposta que muda a cada chamada.

## Duas adaptações minhas, e por quê

**1. `AgentExecutor` e `create_tool_calling_agent` não existem mais.** Foram removidos no LangChain 1.x. O `pyproject.toml` deste curso fixa `langchain>=1.4`, onde a API é `create_agent`. Ou seja: o código dos vídeos 02 e 03 não roda na versão que o próprio curso instala — e isso não tem relação com Python ou com chave de API. Migrei, e mostro o original lado a lado.

**2. O Tavily do video03 exige chave paga de terceiro.** Troquei por uma ferramenta de busca local, marcada `# [meu]`. A estrutura do agente é idêntica; só a fonte dos dados muda.

Quando você tiver uma `GOOGLE_API_KEY`, o caminho de volta é trocar o modelo falso pelo `ChatGoogleGenerativeAI` — uma linha por célula. Está indicado em cada uma.

In [ ]:
# [meu] Modelo falso — substitui a chamada de API. Não faz parte do curso.
#
# Os modelos falsos do langchain_core herdam `bind_tools` da classe base,
# que levanta NotImplementedError. O agente precisa desse método, então a
# subclasse abaixo o implementa devolvendo o próprio modelo.
from langchain_core.language_models import FakeListChatModel, FakeMessagesListChatModel
from langchain_core.messages import AIMessage, HumanMessage


class ModeloFalso(FakeMessagesListChatModel):
    """Devolve mensagens roteirizadas, em ordem, sem rede e sem chave."""

    def bind_tools(self, tools, **kwargs):
        return self


print('ModeloFalso pronto — nenhuma chamada de rede será feita neste notebook.')

---
# Bloco 1 · video01 — LCEL, a chain básica

## Camada 1: o template sozinho, sem modelo nenhum

Antes de ligar qualquer modelo: um `ChatPromptTemplate` é só uma função que transforma variáveis em uma lista de mensagens. Dá para rodar e inspecionar o resultado — isso é 100% real, sem nada falso.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente técnico especialista em arquitetura de software de Inteligência Artificial. Seja direto e profissional."),
    ("user", "Explique brevemente o conceito de {topico} e por que ele é modular.")
])

# O template sozinho já é executável — não precisa de modelo nem de chave
mensagens = prompt_template.invoke({"topico": "LangChain Expression Language (LCEL)"})
for m in mensagens.messages:
    print(f'{type(m).__name__:15s} | {m.content}')

## Camada 2: a chain completa

Código do video01, com uma linha trocada: onde estava `ChatGoogleGenerativeAI(...)`, está o modelo falso.

```python
# original do curso:
# model = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0)
```

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# model = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0)  # original
model = FakeListChatModel(responses=[                                            # [meu]
    "LCEL é uma linguagem declarativa para compor componentes do LangChain com o "
    "operador |. É modular porque cada peça implementa a mesma interface Runnable, "
    "então qualquer uma pode ser trocada sem alterar as outras."
])

output_parser = StrOutputParser()

chain = prompt_template | model | output_parser

topico_exemplo = "LangChain Expression Language (LCEL)"
print(f"[Executando Chain] Solicitando explicação sobre: {topico_exemplo}...\n")

resposta = chain.invoke({"topico": topico_exemplo})

print("=== RESPOSTA DA CHAIN ===")
print(resposta)
print()
print('tipo do que saiu:', type(resposta).__name__)

In [ ]:
# [meu] apoio: a mesma chain, sem o parser no fim
chain_sem_parser = prompt_template | FakeListChatModel(responses=["qualquer coisa"])
saida = chain_sem_parser.invoke({"topico": "LCEL"})
print('sem StrOutputParser →', type(saida).__name__)
print('conteúdo            →', saida.content)
print()
print('cada elo da chain:', [type(p).__name__ for p in chain.steps])

### Sua vez

1. Rode a célula de apoio acima. Sem o `StrOutputParser` sai um objeto; com ele sai outra coisa. Qual é a diferença, e por que a chain não devolve texto puro por padrão?
2. O operador `|` é o mesmo do pipe do shell. Os três elos da chain são objetos de tipos diferentes, mas encaixam. O que eles precisam ter em comum para isso funcionar — e que nome o LangChain dá a essa interface?
3. O `temperature=0` do original foi para o modelo, não para o template nem para o parser. Depois de ver os três elos, diga qual deles é o único **não determinístico** num cenário real, e o que isso implica para testar uma chain.

_

_

---
# Bloco 2 · video02 — Ferramentas e agente

## Camada 1: a ferramenta, sem agente e sem modelo

O decorador `@tool` transforma uma função Python comum em algo que um modelo consegue enxergar: nome, descrição e schema de argumentos. Tudo isso é inspecionável e executável sem nenhuma LLM envolvida.

**Repare na docstring.** Ela não é documentação para humano — é o texto que o modelo lê para decidir se chama a função. É parte do prompt, não do código.

In [ ]:
from langchain_core.tools import tool

@tool
def calcular_custo_tokens(quantidade_tokens: int, modelo: str) -> str:
    """Calcula o custo estimado em dólares com base na quantidade de tokens e no modelo especificado."""
    modelo = modelo.lower()
    if "gpt-4o-mini" in modelo:
        custo = (quantidade_tokens / 1_000_000) * 0.15 # Exemplo de preço hipotético por milhão
        return f"O custo estimado para {quantidade_tokens} tokens no gpt-4o-mini é de ${custo:.6f} USD."
    else:
        return "Modelo não mapeado para cálculo de custo local."

# O que o modelo enxerga desta função:
print('nome      :', calcular_custo_tokens.name)
print('descrição :', calcular_custo_tokens.description)
print('argumentos:', calcular_custo_tokens.args)
print()
# E dá para executar direto, sem modelo nenhum:
print('execução direta →', calcular_custo_tokens.invoke(
    {"quantidade_tokens": 5_000_000, "modelo": "gpt-4o-mini"}))

### Sua vez

1. O `args` foi montado sozinho. De onde o LangChain tirou os tipos `integer` e `string`, já que ninguém os declarou em lugar nenhum além da assinatura?
2. A docstring vira a `description` que o modelo lê. Reescreva a docstring desta função de um jeito que faria o modelo **deixar de chamá-la** quando deveria — e diga o que isso ensina sobre escrever ferramentas.
3. A função devolve uma **string**, não um número. Por quê? Pense em quem é o consumidor do retorno.

_

_

## Camada 3: o agente

> ### ⚠️ Aqui o código do curso não roda — e não é por falta de chave
>
> O video02 original faz assim:
>
> ```python
> from langchain.agents import AgentExecutor, create_tool_calling_agent
>
> prompt = ChatPromptTemplate.from_messages([
>     ("system", "Você é um agente financeiro de IA. ..."),
>     MessagesPlaceholder(variable_name="chat_history"),
>     ("user", "{input}"),
>     MessagesPlaceholder(variable_name="agent_scratchpad"),
> ])
> agent = create_tool_calling_agent(llm, tools, prompt)
> agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
> agent_executor.invoke({"input": ..., "chat_history": ...})
> ```
>
> `AgentExecutor` e `create_tool_calling_agent` **foram removidos** no LangChain 1.x. O `langchain.agents` instalado exporta apenas `create_agent`, `AgentState`, `middleware` e `structured_output`.
>
> **A migração não é um rename.** Três coisas mudam de lugar:
> - o `system` vira o parâmetro `system_prompt`, uma string — não mais um `ChatPromptTemplate`;
> - o `MessagesPlaceholder("agent_scratchpad")` **desaparece** — o grafo cuida disso internamente;
> - o `chat_history` deixa de ser um placeholder e passa a ser simplesmente as mensagens anteriores dentro de `messages`.

O roteiro abaixo diz ao modelo falso o que "decidir": primeiro pedir a ferramenta, depois responder. **A ferramenta executa de verdade** — o valor em dólar é conta do Python.

In [ ]:
from langchain.agents import create_agent   # [meu] substitui AgentExecutor + create_tool_calling_agent

tools = [calcular_custo_tokens]

# [meu] roteiro do modelo falso: 1º turno pede a ferramenta, 2º turno responde
roteiro = [
    AIMessage(content="", tool_calls=[{
        "name": "calcular_custo_tokens",
        "args": {"quantidade_tokens": 5_000_000, "modelo": "gpt-4o-mini"},
        "id": "call_1",
    }]),
    AIMessage(content="Rodar 5.000.000 de tokens no gpt-4o-mini custa cerca de US$ 0,75."),
]
llm = ModeloFalso(responses=roteiro)   # no lugar de ChatGoogleGenerativeAI(...)

agente = create_agent(
    llm,
    tools,
    system_prompt="Você é um agente financeiro de IA. Use as ferramentas disponíveis para responder dúvidas sobre custos se necessário.",
)

historico_conversa = [
    HumanMessage(content="Olá, meu nome é Carlos e estou construindo um assistente."),
    AIMessage(content="Olá Carlos! Como posso te ajudar com os custos ou arquitetura do seu assistente hoje?"),
]
pergunta_usuario = "Se meu sistema rodar 5.000.000 de tokens no gpt-4o-mini, quanto vou gastar?"

resultado = agente.invoke({
    "messages": historico_conversa + [HumanMessage(content=pergunta_usuario)]
})

print("=== O LOOP DO AGENTE, PASSO A PASSO ===")
for m in resultado["messages"]:
    chamadas = getattr(m, "tool_calls", None)
    if chamadas:
        print(f'{type(m).__name__:13s} | PEDE FERRAMENTA → {chamadas[0]["name"]}({chamadas[0]["args"]})')
    else:
        print(f'{type(m).__name__:13s} | {m.content}')

### Sua vez

1. Conte as mensagens da saída e identifique quem produziu cada uma. Em quantas **idas ao modelo** esse agente respondeu uma única pergunta? Qual a consequência disso em custo e latência num sistema real?
2. O `ToolMessage` traz o valor calculado, e a `AIMessage` final o repete em texto. Por que o agente precisa de um segundo turno em vez de devolver o resultado da ferramenta direto ao usuário?
3. O original passava `chat_history` por um `MessagesPlaceholder` e a versão nova passa tudo dentro de `messages`. Olhando a saída, o histórico do Carlos chegou até o modelo? Como você comprova isso a partir do que foi impresso?

_

_

---
# Bloco 3 · video03 — Answer Engine

> ### ⚠️ Duas adaptações nesta seção
>
> **1.** O `TavilyAnswer` exige `TAVILY_API_KEY`, chave de terceiro. Substituí por uma ferramenta de busca local com resultados fixos, marcada `# [meu]`. A assinatura e o papel no agente são os mesmos.
>
> **2.** Mesma migração de `AgentExecutor` para `create_agent` do bloco anterior. O `handle_parsing_errors=True` do original não tem equivalente direto: no grafo, tratamento de erro é feito por middleware.

O ponto deste vídeo é o padrão, não a ferramenta: **o modelo não sabe a resposta e vai buscá-la fora**. É o mesmo desenho do seu Tech Challenge — lá a fonte é o índice das reviews do Olist em vez da internet.

In [ ]:
# [meu] substitui o TavilyAnswer, que exige TAVILY_API_KEY
@tool
def buscar_na_internet(consulta: str) -> str:
    """Busca informações atuais na internet sobre fatos recentes, notícias, esportes e cotações."""
    base = {
        "fórmula 1": "O campeonato de Fórmula 1 de 2025 foi vencido por Lando Norris (McLaren), "
                     "com 423 pontos contra 408 de Max Verstappen.",
    }
    for chave, valor in base.items():
        if chave in consulta.lower():
            return valor
    return "Nenhum resultado encontrado para essa consulta."

ferramentas_projeto = [buscar_na_internet]

# [meu] roteiro: o modelo busca, depois resume citando a fonte
roteiro_engine = [
    AIMessage(content="", tool_calls=[{
        "name": "buscar_na_internet",
        "args": {"consulta": "quem ganhou o último campeonato de fórmula 1"},
        "id": "call_busca",
    }]),
    AIMessage(content="Segundo a busca, Lando Norris (McLaren) venceu o campeonato de 2025, "
                      "com 423 pontos contra 408 de Max Verstappen."),
]

answer_engine = create_agent(
    ModeloFalso(responses=roteiro_engine),
    ferramentas_projeto,
    system_prompt=(
        "Você é o 'Answer Engine', um motor de respostas avançado integrado à internet. "
        "Sempre que o usuário perguntar sobre fatos recentes, ações de mercado, notícias ou cotações, "
        "use a ferramenta de busca para trazer dados verídicos e cite as descobertas de forma resumida."
    ),
)

pergunta_atual = "Quem ganhou o último campeonato de Fórmula 1 e qual foi a pontuação final?"
resultado_engine = answer_engine.invoke({"messages": [HumanMessage(content=pergunta_atual)]})

print("=== ANSWER ENGINE, PASSO A PASSO ===")
for m in resultado_engine["messages"]:
    chamadas = getattr(m, "tool_calls", None)
    if chamadas:
        print(f'{type(m).__name__:13s} | BUSCA → {chamadas[0]["args"]["consulta"]}')
    else:
        print(f'{type(m).__name__:13s} | {m.content}')

### Sua vez

1. O prompt de sistema manda "trazer dados verídicos e citar as descobertas". Olhe a resposta final: ela **cita** a fonte? O que no desenho garante que ela não invente — e o que **não** garante nada?
2. Experimente trocar a pergunta para algo que a ferramenta não conhece, e veja o que a função devolve. Num sistema real, o que deveria acontecer quando a busca volta vazia, e o que um modelo tende a fazer se nada o impedir?
3. Compare os dois agentes do notebook. O do bloco 2 chama uma ferramenta que **calcula**; este chama uma que **recupera**. Qual dos dois se parece com o seu Tech Challenge, e o que isso muda em relação a confiar na resposta?

_

_

---
## Fechamento — responda sem olhar para cima

1. Em uma frase cada: o que é uma **chain** e o que é um **agente**. A diferença decisiva está em quem toma uma decisão — qual decisão, e quem a toma em cada caso?
2. Nos três vídeos o modelo nunca executou nada. Quem executa a ferramenta? Descreva o caminho completo, do pedido do modelo até o valor voltar para ele.
3. Esta aula inteira rodou com o modelo falso. O que você **não** pode concluir por causa disso — e qual é a primeira coisa que você testaria assim que tivesse uma chave?

_

_

## Ponte para o Tech Challenge

4. O Answer Engine do bloco 3 é a arquitetura do seu desafio com outra fonte. Escreva o desenho do seu sistema nos termos desta aula: qual é a ferramenta, o que ela recupera, e o que entra no `system_prompt`.
5. O requisito 4 do seu Tech Challenge é **abstenção**: reconhecer quando não há evidência e não responder. Na pergunta 2 do bloco 3 você viu a busca voltar vazia. Onde, nessa arquitetura, você implementaria a abstenção — no prompt, na ferramenta, ou num terceiro lugar? Justifique; essa decisão entra no README.

_

_